## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
from openai import OpenAI
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


In [4]:
instructions1 = "Usted es un agente de ventas que trabaja para ComplAI, \
una empresa que proporciona una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos serios y profesionales."

instructions2 = "Eres un agente de ventas divertido y atractivo que trabaja para ComplAI, \
una empresa que proporciona una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos en frío ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Usted es un agente de ventas muy ocupado que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos concisos y directos."

### It's easy to use any models with OpenAI compatible endpoints

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [6]:

openai_client = AsyncOpenAI(api_key=openai_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

openai_model = OpenAIChatCompletionsModel(model="gpt-4.1-mini", openai_client=openai_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
lama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [7]:
sales_agent1 = Agent(name="OpenAI Sales Agent", instructions=instructions1, model=openai_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=lama3_3_model)

In [8]:
description = "Escribir un correo electrónico de venta en frío"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

# ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# model_name = "deepseek-r1:8b"
# message = [{"role": "user", "content": "Respóndeme bonito; es una prueba."}]
# response = ollama.chat.completions.create(model=model_name, messages=message)
# answer = response.choices[0].message.content

In [9]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envíe un correo electrónico con el asunto y el cuerpo HTML indicados a todos los clientes potenciales. """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dnaranjo88@gmail.com")  # Change to your verified sender
    to_email = To("zews.kevina@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [10]:
subject_instructions = "Puedes escribir un asunto para un correo electrónico de ventas en frío. \
Te dan un mensaje y tienes que escribir un asunto para un correo electrónico que probablemente obtenga respuesta."

html_instructions = "Puede convertir un cuerpo de correo electrónico de texto en un cuerpo de correo electrónico HTML. \
Se le da un cuerpo de correo electrónico de texto que podría tener algunos markdown \
y tiene que convertirlo en un cuerpo de correo electrónico HTML con un diseño simple, claro y convincente."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [11]:
email_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="Usted es un formateador y remitente de correo electrónico. Recibe el cuerpo de un correo electrónico para enviarlo. \
Primero utiliza la herramienta subject_writer para escribir un asunto para el correo electrónico, luego utiliza la herramienta html_converter para convertir el cuerpo a HTML. \
Por último, utilice la herramienta send_html_email para enviar el correo electrónico con el asunto y el cuerpo HTML. "


c

In [14]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [15]:
sales_manager_instructions = """
Eres Director de Ventas en ComplAI. Su objetivo es encontrar el mejor correo electrónico de ventas en frío utilizando las herramientas sales_agent.
 
Siga estos pasos cuidadosamente:
1. 1. Genere borradores: Utilice las tres herramientas sales_agent para generar tres borradores de correo electrónico diferentes. No continúe hasta que los tres borradores estén listos.
 
2. 2. Evaluar y seleccionar: Revise los borradores y elija el mejor correo electrónico según su criterio.
Puedes utilizar las herramientas varias veces si no estás satisfecho con los resultados del primer intento.
 
3. 3. Entrega para el envío: Pase ÚNICAMENTE el borrador del email ganador al agente 'Email Manager'. El Email Manager se encargará del formateo y del envío.
 
Reglas cruciales:
- Debe utilizar las herramientas del agente de ventas para generar los borradores - no los escriba usted mismo.
- Debe entregar exactamente UN email al Email Manager - nunca más de uno.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Envía un correo electrónico de ventas en frío dirigido a Dear CEO from Kevin"

with trace("SDR automatizado - Kevin"):
    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [16]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Nombre check",
    instructions="Comprueba si el usuario está incluyendo el nombre personal de alguien en lo que quiere que hagas.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [18]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [19]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Envía un correo electrónico de ventas en frío dirigido a Dear CEO from Kevin"

with trace("Protected Automated SDR - Kevin"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [20]:

message = "Enviar un correo electrónico de ventas en frío dirigido a Dear CEO from Head of Business Development."

with trace("Protected Automated SDR - kevin"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">- Pruebe diferentes modelos<br/>- Añada más barandillas de entrada y salida<br/>- Utilice salidas estructuradas para la generación de correo electrónico.
            </span>
        </td>
    </tr>
</table>